In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Ketaksamaan CHSH

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Anggaran penggunaan: Dua minit pada pemproses Heron r3 (NOTA: Ini adalah anggaran sahaja. Masa jalan sebenar mungkin berbeza.)*
## Hasil pembelajaran
Selepas menamatkan tutorial ini, kamu dijangka memahami perkara berikut:
- Cara membina Circuit Bell-state CHSH berparameter dan mengukur empat nilai jangkaan yang membentuk saksi CHSH.
- Cara mengira nilai jangkaan bagi pelbagai boleh ukur dalam satu sapuan parameter menggunakan satu panggilan kepada primitif [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2).
- Cara mengesahkan aliran kerja kuantum pada simulator setempat yang bising dengan `AerSimulator.from_backend` sebelum menghantar ke perkakasan.
- Cara menskalakan eksperimen CHSH kepada penanda aras keterjaitan seluruh peranti dengan menjalankan banyak pasangan Bell bebas secara selari pada perkakasan IBM Quantum&reg;.

## Prasyarat
Adalah disyorkan supaya kamu membiasakan diri dengan topik-topik ini:
- [Keterjaitan dalam Tindakan](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game), sebuah pelajaran kursus tentang keadaan Bell dan permainan CHSH.
- [`SparsePauliOp`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) dan [pengenalan](/guides/primitives) kepada primitif Qiskit.
## Latar belakang
Dalam tutorial ini, kamu akan menjalankan eksperimen pada komputer kuantum untuk menunjukkan pelanggaran ketaksamaan CHSH menggunakan primitif Estimator.

Ketaksamaan CHSH, dinamakan sempena Clauser, Horne, Shimony, dan Holt, digunakan untuk menguji teorem Bell secara eksperimental (1969). Teorem ini menyatakan bahawa teori pembolehubah tersembunyi setempat tidak dapat menjelaskan beberapa kesan keterjaitan dalam mekanik kuantum. Menunjukkan pelanggaran ketaksamaan CHSH membuktikan bahawa mekanik kuantum tidak serasi dengan teori pembolehubah tersembunyi setempat — satu eksperimen yang menjadi asas kepada pemahaman kita tentang mekanik kuantum.

Hadiah Nobel Fizik 2022 telah dianugerahkan kepada Alain Aspect, John Clauser, dan Anton Zeilinger sebahagiannya atas kerja perintis mereka dalam sains maklumat kuantum, khususnya eksperimen mereka dengan foton terjalin yang menunjukkan pelanggaran ketaksamaan Bell.

Untuk eksperimen ini, kita akan mencipta pasangan terjalin di mana kita mengukur setiap qubit dalam dua asas yang berbeza. Kita akan melabelkan asas untuk qubit pertama sebagai $A$ dan $a$ serta asas untuk qubit kedua sebagai $B$ dan $b$. Ini membolehkan kita mengira kuantiti CHSH $S_1$:

$$
S_1 = A(B-b) + a(B+b).
$$

Setiap boleh ukur bernilai sama ada $+1$ atau $-1$. Jelas sekali, salah satu daripada sebutan $B\pm b$ mesti $0$, dan yang satu lagi mesti $\pm 2$. Oleh itu, $S_1 = \pm 2$. Nilai purata $S_1$ mesti memenuhi ketaksamaan:

$$
|\langle S_1 \rangle|\leq 2.
$$

Mengembangkan $S_1$ dalam sebutan $A$, $a$, $B$, dan $b$ menghasilkan:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2.
$$

Kamu boleh mentakrifkan kuantiti CHSH lain $S_2$:

$$
S_2 = A(B+b) - a(B-b),
$$

yang membawa kepada ketaksamaan lain:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2.
$$

Sekiranya mekanik kuantum boleh diterangkan oleh teori pembolehubah tersembunyi setempat, ketaksamaan-ketaksamaan ini akan sentiasa terpenuhi. Seperti yang ditunjukkan dalam tutorial ini, ketaksamaan-ketaksamaan ini boleh dilanggar pada komputer kuantum, membuktikan bahawa mekanik kuantum tidak serasi dengan teori pembolehubah tersembunyi setempat.

Kita mencipta pasangan terjalin dengan menyediakan keadaan Bell $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$. Menggunakan primitif Estimator, kita mendapatkan nilai jangkaan $\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$, dan $\langle ab \rangle$ secara terus, tanpa perlu membinanya semula daripada kiraan mentah. Kita mengukur qubit kedua dalam asas $Z$ dan $X$. Qubit pertama juga diukur dalam asas ortogon, tetapi dengan sudut putaran $\theta$ yang kita sapu antara $0$ dan $2\pi$. Primitif Estimator menilai sapuan parameter ini dalam satu [blok primitif bersatu (PUB)](/guides/primitive-input-output) tunggal.
## Keperluan
Sebelum memulakan tutorial ini, pastikan kamu telah memasang yang berikut:

- Qiskit SDK v2.0 atau lebih baru, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.40 atau lebih baru (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.17 atau lebih baru (`pip install qiskit-aer`)
## Persediaan

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Qiskit Aer for local noisy simulation
from qiskit_aer import AerSimulator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

In [ ]:
# Select an IBM Quantum backend.
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=127, operational=True, simulator=False
)
backend.name

'ibm_pittsburgh'

## Small-scale simulator example

Before submitting a hardware job, we validate the entire workflow on a local noisy simulator. We use `AerSimulator.from_backend(backend)` to build a simulator that inherits the noise model and coupling map of the backend you selected, so the simulator response is qualitatively similar to what we expect from hardware.

### Step 1: Map classical inputs to a quantum problem

We write the CHSH circuit with a single parameter $\theta$, which sweeps the measurement basis of the first qubit. The [`Estimator`](/docs/api/qiskit-ibm-runtime/estimator-v2) primitive simplifies the analysis: it returns expectation values of observables directly, and it can evaluate a parameterized circuit at many parameter values in a single call.

In [3]:
theta = Parameter(r"$\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/c3f57d25-0.avif" alt="Output of the previous code cell" />

## Contoh simulator berskala kecil
Sebelum menghantar kerja perkakasan, kita mengesahkan keseluruhan aliran kerja pada simulator setempat yang bising. Kita menggunakan `AerSimulator.from_backend(backend)` untuk membina simulator yang mewarisi model hingar dan peta gandingan backend yang kamu pilih, supaya respons simulator secara kualitatif serupa dengan apa yang kita jangkakan daripada perkakasan.
### Langkah 1: Petakan input klasik kepada masalah kuantum
Kita menulis Circuit CHSH dengan satu parameter $\theta$, yang menyapu asas pengukuran qubit pertama. Primitif [`Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) memudahkan analisis: ia memulangkan nilai jangkaan boleh ukur secara terus, dan boleh menilai Circuit berparameter pada banyak nilai parameter dalam satu panggilan.

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as a list of lists for the Estimator PUB
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/c3f57d25-0.avif)

Seterusnya, kita mencipta senarai 21 nilai fasa dari $0$ hingga $2\pi$ untuk menilai Circuit berparameter ($0$, $0.1\pi$, $0.2\pi$, ..., $1.9\pi$, $2\pi$).

In [5]:
# <S_1> = <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <S_2> = <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

Akhirnya kita mentakrifkan boleh ukur. Qubit pertama diukur sepanjang paksi yang diputar sebanyak $\theta$; qubit kedua diukur dalam $Z$ dan $X$. Dengan pilihan tersebut, empat perekat CHSH dipetakan kepada operator Pauli $ZZ$, $ZX$, $XZ$, dan $XX$:

$$
\langle S_1 \rangle = \langle ZZ \rangle - \langle ZX \rangle + \langle XZ \rangle + \langle XX \rangle,
$$

$$
\langle S_2 \rangle = \langle ZZ \rangle + \langle ZX \rangle - \langle XZ \rangle + \langle XX \rangle.
$$

In [6]:
# Build a noisy simulator from the ibm_pittsburgh backend
aer_sim = AerSimulator.from_backend(backend)

pm = generate_preset_pass_manager(target=aer_sim.target, optimization_level=3)
chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/3e139a89-0.avif" alt="Output of the previous code cell" />

### Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Primitif V2 hanya menerima Circuit dan boleh ukur yang mematuhi arahan dan ketersambungan yang disokong oleh sistem sasaran (seni bina set arahan, atau ISA, Circuit dan boleh ukur). Kita membina `AerSimulator` daripada backend dan melakukan transpilasi terhadap sasaran simulator supaya pengurus laluan yang sama dilaksanakan dari hujung ke hujung.

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/3e139a89-0.avif)

Kita juga mengubah boleh ukur supaya sepadan dengan susun atur qubit Circuit yang ditranspil menggunakan `SparsePauliOp.apply_layout`.

In [8]:
# Use the AerSimulator-backed Estimator to validate the workflow locally
estimator_sim = Estimator(mode=aer_sim)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA observables
    individual_phases,  # Parameter values
)

sim_result = estimator_sim.run(pubs=[pub]).result()

### Langkah 3: Laksanakan menggunakan primitif Qiskit
Jalankan sapuan parameter dengan `EstimatorV2` dalam mod `aer_sim`. Kaedah `run()` Estimator mengambil iterable bagi PUB. Setiap PUB mempunyai format `(circuit, observables, parameter_values, precision)`. Kita menghantar kedua-dua boleh ukur bersama supaya ia berkongsi sapuan parameter yang sama.

In [9]:
chsh1_sim = sim_result[0].data.evs[0]
chsh2_sim = sim_result[0].data.evs[1]


def plot_chsh(phases, chsh1, chsh2, title):
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(
        phases / np.pi, chsh1, "o-", label=r"$\langle S_1 \rangle$", zorder=3
    )
    ax.plot(
        phases / np.pi, chsh2, "o-", label=r"$\langle S_2 \rangle$", zorder=3
    )

    # classical bound +-2
    ax.axhline(y=2, color="0.9", linestyle="--")
    ax.axhline(y=-2, color="0.9", linestyle="--")

    # quantum bound, +-2*sqrt(2)
    ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
    ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
    ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
    ax.fill_between(
        phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7
    )

    ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
    ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel("CHSH witness")
    ax.set_title(title)
    ax.legend()
    plt.show()


plot_chsh(
    phases,
    chsh1_sim,
    chsh2_sim,
    "CHSH witnesses from AerSimulator (ibm_pittsburgh noise model)",
)

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/c8fd5140-0.avif" alt="Output of the previous code cell" />

### Langkah 4: Proses pasca dan kembalikan hasil dalam format klasik yang dikehendaki
Estimator memulangkan nilai jangkaan bagi kedua-dua boleh ukur. Kita memplotnya berbanding $\theta$ bersama-sama sempadan klasik ($\pm 2$) dan sempadan Tsirelson ($\pm 2\sqrt{2}$). Kawasan kelabu berlorek menandai jurang antara keduanya. Titik yang jatuh dalam jalur ini melanggar ketaksamaan CHSH.

In [10]:
# -------------------------Step 1: Map classical inputs to a quantum problem-------------------------
# A CHSH test is bipartite, so we scale up by running one independent CHSH
# experiment on every disjoint Bell pair the device can host. A greedy
# matching of the coupling map gives a set of edges that share no qubits.
num_qubits = backend.num_qubits
used = set()
pairs = []
for qa, qb in backend.coupling_map.get_edges():
    if qa not in used and qb not in used:
        pairs.append((qa, qb))
        used.update((qa, qb))
num_pairs = len(pairs)
print(
    f"Tiling {backend.name} with {num_pairs} parallel Bell pairs "
    f"({2 * num_pairs} of {num_qubits} qubits)"
)

# One parameterized CHSH sub-circuit per pair, all sharing the angle theta
theta = Parameter(r"$\theta$")
chsh_circuit = QuantumCircuit(num_qubits)
for qa, qb in pairs:
    chsh_circuit.h(qa)
    chsh_circuit.cx(qa, qb)
    chsh_circuit.ry(theta, qa)

# Embed the two CHSH observables onto each pair's qubits (identity elsewhere)
obs1 = SparsePauliOp.from_list([("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)])
obs2 = SparsePauliOp.from_list([("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)])
observables = []
for qa, qb in pairs:
    observables.append([obs1.apply_layout([qa, qb], num_qubits)])
    observables.append([obs2.apply_layout([qa, qb], num_qubits)])

number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
individual_phases = [[ph] for ph in phases]

# -------------------------Step 2: Optimize problem for quantum hardware execution-------------------------
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
chsh_isa_circuit = pm.run(chsh_circuit)
isa_observables = [
    [o[0].apply_layout(chsh_isa_circuit.layout)] for o in observables
]

# -------------------------Step 3: Execute using Qiskit primitives-------------------------
estimator_hw = Estimator(mode=backend)
estimator_hw.options.environment.job_tags = ["TUT_CI"]

pub = (chsh_isa_circuit, isa_observables, individual_phases)
job = estimator_hw.run(pubs=[pub])
print(f"Job ID: {job.job_id()}")
hw_result = job.result()

# -------------------------Step 4: Post-process and return result in desired classical format-------------------------
# evs has shape (2 * num_pairs, number_of_phases); rows alternate S1, S2
evs = np.asarray(hw_result[0].data.evs)
chsh1_all = evs[0::2]
chsh2_all = evs[1::2]

# A pair "violates" CHSH if its strongest witness exceeds the classical bound
peak = np.maximum(
    np.abs(chsh1_all).max(axis=1), np.abs(chsh2_all).max(axis=1)
)
n_violate = int(np.sum(peak > 2))
print(
    f"{n_violate}/{num_pairs} Bell pairs violated the CHSH inequality "
    f"(mean peak witness {peak.mean():.2f}, classical bound 2)"
)

fig, ax = plt.subplots(figsize=(10, 6))

# Faint individual per-pair curves
for row in chsh1_all:
    ax.plot(phases / np.pi, row, color="#1f77b4", alpha=0.2, lw=1)
for row in chsh2_all:
    ax.plot(phases / np.pi, row, color="#ff7f0e", alpha=0.2, lw=1)

# Bold mean curves across all pairs
ax.plot(
    phases / np.pi,
    chsh1_all.mean(axis=0),
    color="#1f77b4",
    lw=2.5,
    label=r"$\langle S_1 \rangle$ (mean)",
)
ax.plot(
    phases / np.pi,
    chsh2_all.mean(axis=0),
    color="#ff7f0e",
    lw=2.5,
    label=r"$\langle S_2 \rangle$ (mean)",
)

# classical bound +-2 and Tsirelson bound +-2*sqrt(2)
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))
ax.set_xlabel(r"$\theta$")
ax.set_ylabel("CHSH witness")
ax.set_title(
    f"CHSH witnesses for {num_pairs} parallel Bell pairs on {backend.name}"
)
ax.legend()
plt.show()

Tiling ibm_pittsburgh with 64 parallel Bell pairs (128 of 156 qubits)
Job ID: d86efd5g7okc73el0rp0
63/64 Bell pairs violated the CHSH inequality (mean peak witness 2.75, classical bound 2)


<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/3376bc73-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/c8fd5140-0.avif)

Saksi CHSH simulator sudah melebihi sempadan klasik $\pm 2$ pada beberapa nilai $\theta$, walaupun dengan model hingar backend. Puncak-puncak sedikit kurang daripada sempadan Tsirelson $\pm 2\sqrt{2}$ disebabkan oleh hingar peranti yang disimulasikan. Dengan aliran kerja yang telah disahkan, kita beralih kepada perkakasan sebenar.

## Contoh perkakasan berskala besar

Ujian CHSH pada asasnya adalah eksperimen *dua-qubit*, jadi ia tidak berskala dengan membuat satu Circuit yang lebih besar. Sebaliknya, ia berskala dengan menjalankan **banyak ujian secara selari**. Di sini kita merubin Backend dengan seberapa banyak pasangan Bell yang tidak bertindih seperti yang dibenarkan oleh ketersambungannya (satu *padanan* peta gandingan) dan menjalankan sub-Circuit CHSH bebas pada setiap pasangan, semuanya dalam satu kerja tunggal.

Ini mengubah CHSH menjadi **penanda aras keterjaitan seluruh peranti**: daripada satu pasangan yang dipilih tangan, kita menguji keterjaitan merentas sebahagian besar cip sekaligus, dalam keadaan realistik di mana setiap pasangan bersaing dengan silang-ganggu jiran dan ralat get selari. Melanggar ketaksamaan pada setiap pasangan *serentak* mengesahkan bahawa keterjaitan tulen tersedia di seluruh peranti.